# The WSP Heuristic

Here we calculate the WSPD for a given problem and then check whether the number of edges crossing the each pair of clusters is the correct number given the number of exits out of the cluster as per the proved theorems. 

## Using internal python implementation of WSPD class

In [1]:
from pathlib import Path

import tsplib95
import numpy as np

from wsp import ds, tsp

TREE_TYPE = ds.PKPRQuadTree
SIZE_THRESHOLD = 250
TSP_LIB_PATH = Path("TSPLIB")
S_FACTOR = 1.5
#ax = np.array([None, None])
#sys.setrecursionlimit(500_000)

In [2]:
#all_problems : dict[str, tsp.TravellingSalesmanProblem[TREE_TYPE]] = {}

#for file in sorted(TSP_LIB_PATH.iterdir()): # Loop through every tsp
#    if file.suffix != ".tsp" or not file.is_file():
#        continue
#    problem = tsplib95.load(f"{file.absolute()}")
#    if problem.edge_weight_type != "EUC_2D": # Skip non-Euclidean TSPs
#        continue
#    if problem.dimension > SIZE_THRESHOLD: # Skip large TSPs
#        continue

#    points = [ds.Point(*problem.node_coords[i]) for i in problem.get_nodes()]
#    ts_problem = tsp.TravellingSalesmanProblem[TREE_TYPE](TREE_TYPE, points, ax, s=S_FACTOR) 
    
#    all_problems[problem.name] = ts_problem
#    print(f"Added {problem.name}")

#print("Found", len(all_problems), "euclidean TSPs")

In [3]:
#for name, problem in all_problems.items():
#    tour, _, _ = problem.nnn_path
#    badCount = 0
#    for (A, B), _ in problem.wspd:
#        pointsA = set(A.covered_points)
#        pointsB = set(B.covered_points)
#        if not problem.wsp_heuristic_good(tour, pointsA, pointsB):
#            badCount += 1
#    if badCount > 0:
#        print(f"Problem {name} has {badCount} bad pairs")

## Faster Runner using external cpython libs
still solving problem as they are seen

no bad pairs found up to 12k for s=1.5  (TSPLIB EUC2D)
no bad pairs found up to 12k for s=1.0 (TSPLIB EUC2D)

In [4]:
%load_ext line_profiler
from numba import njit
import elkai
from wspd import build_wspd_np, point as wsp_point # uses diam based sep factor
import wspd

SIZE_THRESHOLD2 = 500

In [5]:
all_problems : list[tsplib95.models.StandardProblem] = []

for file in sorted(TSP_LIB_PATH.iterdir()): # Loop through every tsp
    if file.suffix != ".tsp" or not file.is_file():
        continue
    problem = tsplib95.load(f"{file.absolute()}")
    if problem.edge_weight_type != "EUC_2D": # Skip non-Euclidean TSPs
        continue
    if problem.dimension > SIZE_THRESHOLD2 or problem.dimension <= 350: # Skip large TSPs
        continue
    all_problems.append(problem)

len(all_problems)

5

In [6]:
@njit(cache=True)
def wsp_heuristic_good(a_pos, b_pos, num_points: int) -> bool:
    """A heuristic to check if a path is good based on the WSPs

    Takes in the positions of the nodes in A and B in the tour (sorted), and the total number of points in the problem.
    a_pos \subseteq [0, num_points-1] are the positions of the nodes in A in the tour
    b_pos \subseteq [0, num_points-1] are the positions of the nodes in B in the tour

    We then check the following conditions:
    1. If there are no exit pairs with endpoints in separate sets, then there must be exactly two edges connecting A and B
    2. If there are an even number of exit pairs with endpoints in separate sets, then there must be no edges connecting A and B 
    3. If there are an odd number of exit pairs with endpoints in separate sets, then there must be exactly one edge connecting A and B
    """
    assert len(a_pos) > 0 and len(b_pos) > 0, "Sets must be non-empty"

    # keep track of how many edges directly connect A and B in the tour
    biconn_AB_count = 0
    biconn_BA_count = 0
    # keep track of how many exit pairs have endpoints in separate sets
    exit_AA_count = 0
    exit_BB_count = 0
    exit_AB_count = 0
    exit_BA_count = 0

    # check the edge cases
    if a_pos[-1] == num_points - 1 and b_pos[0] == 0:
        biconn_AB_count += 1
    elif b_pos[-1] == num_points - 1 and a_pos[0] == 0:
        biconn_BA_count += 1
    elif (a_pos[-1] == num_points - 1 and a_pos[0] == 0) or (b_pos[-1] == num_points - 1 and b_pos[0] == 0):
        pass # then there is just an a loop and nothing to see here
    elif a_pos[-1] > b_pos[-1]:
        if a_pos[0] < b_pos[0]:
            exit_AA_count += 1
        else:
            exit_AB_count += 1
    else:
        if b_pos[0] < a_pos[0]:
            exit_BB_count += 1
        else:
            exit_BA_count += 1

    # two pointer approach to count biconns and exits
    i = j = 0
    na = len(a_pos)
    nb = len(b_pos)

    # tracking info
    next_a = a_pos[0]
    next_b = b_pos[0]
    exited_A = None

    while True:
        if next_a <= next_b:
            # handle if exited
            if exited_A is not None:
                if exited_A:
                    exit_AA_count += 1
                else:
                    exit_BA_count += 1
                exited_A = None

            # check if biconn or exit
            if next_a + 1 == next_b:
                biconn_AB_count += 1
            elif i + 1 >= na or next_a + 1 != a_pos[i+1]:
                exited_A = True

            # increment i and update next_a
            i += 1
            if i < na:
                next_a = a_pos[i]
            else:
                break
        else:
            # handle if exited
            if exited_A is not None:
                if exited_A:
                    exit_AB_count += 1
                else:
                    exit_BB_count += 1
                exited_A = None

            # check if biconn or exit
            if next_b + 1 == next_a:
                biconn_BA_count += 1
            elif j + 1 >= nb or next_b + 1 != b_pos[j+1]:
                exited_A = False
            
            # increment j and update next_b
            j += 1
            if j < nb:
                next_b = b_pos[j]
            else:
                break
    # at this point one or both sets are expended
    while i < na:
        if exited_A is not None:
            if exited_A:
                exit_AA_count += 1
            else:
                exit_BA_count += 1
            exited_A = None
        if i + 1 >= na or a_pos[i] + 1 != a_pos[i+1]:
            exited_A = True
        i += 1
    while j < nb:
        if exited_A is not None:
            if exited_A:
                exit_AB_count += 1
            else:
                exit_BB_count += 1
            exited_A = None
        if j + 1 >= nb or b_pos[j] + 1 != b_pos[j+1]:
            exited_A = False
        j += 1
        

    ## covers both single in outs and multi case
    cross_exits = exit_AB_count + exit_BA_count
    if cross_exits == 0:
        return biconn_AB_count == 1 and biconn_BA_count == 1
    elif cross_exits % 2 == 0:
        return biconn_AB_count == 0 and biconn_BA_count == 0
    else:
        return (biconn_AB_count == 1 and biconn_BA_count == 0) or (biconn_AB_count == 0 and biconn_BA_count == 1)

In [17]:
def check_problem_tour(prob: tsplib95.models.StandardProblem, tour: np.ndarray, dim=2) -> tuple[int, int]:
    """Given a TSP and a wrapping tour, checks if the WSP heuristic holds"""
    points = [wsp_point(prob.node_coords[i]) for i in prob.get_nodes()]

    # Build the WSPDs with different separation factors
    wspd_15: list[tuple[list[int], list[int]]] = build_wspd_np(len(points), dim, 1.5*2, points)
    #print("built wspd1.5", flush=True)

    # pruning problems which cannot be bad
    wspd_15_conv = filter(lambda x: len(x[0]) > 1 and len(x[1]) > 1, wspd_15)

    # vectorized node -> position map
    pos_in_tour = np.empty(len(tour) - 1, dtype=np.uint32)
    pos_in_tour[tour[:-1]] = np.arange(len(tour) - 1, dtype=np.uint32)

    biggest_pair = 0
    bad_count = 0
    for A, B in wspd_15_conv:
        biggest_pair = max(biggest_pair, min(len(A),len(B)))
        pos_a = np.sort(pos_in_tour[A])
        pos_b = np.sort(pos_in_tour[B])
        if not wsp_heuristic_good(pos_a, pos_b, len(tour) - 1):
            bad_count += 1
            #coords = np.asarray([prob.node_coords[i] for i in prob.get_nodes()], dtype=np.float64)

            A_pts = np.asarray([prob.node_coords[i+1] for i in A], dtype=np.float32)
            B_pts = np.asarray([prob.node_coords[i+1] for i in B], dtype=np.float32)

            # exact min inter-cluster distance
            min_dist = np.linalg.norm(
                A_pts[:, None, :] - B_pts[None, :, :], axis=2
            ).min()

            # exact cluster diameters
            diam_A = np.linalg.norm(
                A_pts[:, None, :] - A_pts[None, :, :], axis=2
            ).max()
            diam_B = np.linalg.norm(
                B_pts[:, None, :] - B_pts[None, :, :], axis=2
            ).max()

            true_sep = min_dist / max(diam_A, diam_B)

            print(
                f"{prob.name}: bad pair ({A},{B}), true separation={true_sep:.2f} "
                #f"(min_dist={min_dist:.2f}, diamA={diam_A:.2f}, diamB={diam_B:.2f})"
            )
    return bad_count, biggest_pair

In [8]:
#def profile_bottleneck(all_problems: list[tsplib95.models.StandardProblem]):

#for prob in all_problems:
#    print(f"Checking {prob.name}...", flush=True)

#    lkh_instance = elkai.Coordinates2D({
#        i-1: prob.node_coords[i] for i in prob.get_nodes()
#    })

#    # tour numpy conversions
#    tour_lkh = lkh_instance.solve_tsp(runs=1)
#    tour = np.array(tour_lkh, dtype=np.uint16)

#    check_problem_tour(prob, tour)

#%lprun -f profile_bottleneck -u 1.0 profile_bottleneck(all_problems)
print("Done")

Done


## Loading previously solved TSPs

In [9]:
gaia_prob = tsplib95.load("ALL_tsp/dja1436.tsp")
gaia_tour_prob = tsplib95.load("ALL_tsp/dja1436.opt.tour")

In [10]:
gaia_tour = np.array(gaia_tour_prob.tours[0] + [1], dtype=np.uint32) - 1
gaia_tour.shape

(1437,)

In [11]:
from any_metric_wspd import gen_wspd
from scipy.spatial.distance import pdist, squareform

In [12]:
points = np.asarray([gaia_prob.node_coords[i] for i in gaia_prob.get_nodes()], dtype=np.float32)
dist_matrix = pdist(points, metric="euclidean")
dist_matrix = squareform(dist_matrix)
dist_matrix.shape

(1436, 1436)

In [13]:
wspd = gen_wspd(dist_matrix, 0.9999)
len(wspd)

374367

In [15]:
check_problem_tour(gaia_prob, gaia_tour, dim=2)

#%lprun -f check_problem_tour -u 1.0 check_problem_tour(gaia_prob, gaia_tour, dim=3)

built wspd1.5


(0, 150)

In [16]:
# Scan for all "*.opt.tour" files and test matching EUC_2D / EUC_3D instances
search_roots = [TSP_LIB_PATH, Path("ALL_tsp"), Path(".")]
opt_tour_files = []

seen = set()
for root in search_roots:
    if not root.exists():
        continue
    for p in root.rglob("*.opt.tour"):
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            opt_tour_files.append(p)

opt_tour_files = sorted(opt_tour_files)

results = []
failures = []

for opt_path in opt_tour_files:
    try:
        tsp_path = opt_path.with_suffix("").with_suffix(".tsp")  # *.opt.tour -> *.tsp
        if not tsp_path.exists():
            failures.append((str(opt_path), "matching .tsp not found"))
            continue

        prob = tsplib95.load(str(tsp_path))
        if prob.edge_weight_type not in ("EUC_2D", "EUC_3D"):
            continue

        tour_prob = tsplib95.load(str(opt_path))
        if not getattr(tour_prob, "tours", None):
            failures.append((prob.name, "no tours in opt file"))
            continue

        dim = 2 if prob.edge_weight_type == "EUC_2D" else 3

        # Robust node-id -> index mapping (does not assume nodes are exactly 1..n)
        nodes = list(prob.get_nodes())
        node_to_idx = {node_id: idx for idx, node_id in enumerate(nodes)}

        raw_tour = tour_prob.tours[0]
        try:
            tour_idx = np.array([node_to_idx[n] for n in raw_tour], dtype=np.uint32)
        except KeyError as e:
            failures.append((prob.name, f"tour contains unknown node id: {e}"))
            continue

        # Ensure wrapped/closed tour for check_problem_tour()
        if tour_idx[0] != tour_idx[-1]:
            tour_idx = np.concatenate([tour_idx, tour_idx[:1]])

        print(f"Checking {prob.name} ({prob.edge_weight_type}, n={prob.dimension})...", flush=True)
        bad_count, biggest_pair = check_problem_tour(prob, tour_idx, dim=dim)

        results.append({
            "name": prob.name,
            "type": prob.edge_weight_type,
            "n": prob.dimension,
            "bad_pairs": bad_count,
            "biggest_pair": biggest_pair,
            "opt_tour_file": str(opt_path),
        })

    except Exception as e:
        failures.append((str(opt_path), repr(e)))

print(f"\nChecked {len(results)} EUC_2D/EUC_3D problems with *.opt.tour")
print(f"Failures: {len(failures)}")
if failures:
    print("First 10 failures:")
    for item in failures[:10]:
        print(" -", item)

# Quick aggregate summary
if results:
    total_bad = sum(r["bad_pairs"] for r in results)
    bad_instances = sum(1 for r in results if r["bad_pairs"] > 0)
    print(f"\nTotal bad pairs: {total_bad}")
    print(f"Instances with >=1 bad pair: {bad_instances}/{len(results)}")

Checking Tnm100.tsp (EUC_2D, n=100)...
built wspd1.5
Checking Tnm103.tsp (EUC_2D, n=103)...
built wspd1.5
Checking Tnm106.tsp (EUC_2D, n=106)...
built wspd1.5
Checking Tnm109.tsp (EUC_2D, n=109)...
built wspd1.5
Checking Tnm112.tsp (EUC_2D, n=112)...
built wspd1.5
Checking Tnm115.tsp (EUC_2D, n=115)...
built wspd1.5
Checking Tnm118.tsp (EUC_2D, n=118)...
built wspd1.5
Checking Tnm121.tsp (EUC_2D, n=121)...
built wspd1.5
Checking Tnm124.tsp (EUC_2D, n=124)...
built wspd1.5
Checking Tnm127.tsp (EUC_2D, n=127)...
built wspd1.5
Checking Tnm130.tsp (EUC_2D, n=130)...
built wspd1.5
Checking Tnm133.tsp (EUC_2D, n=133)...
built wspd1.5
Checking Tnm136.tsp (EUC_2D, n=136)...
built wspd1.5
Checking Tnm139.tsp (EUC_2D, n=139)...
built wspd1.5
Checking Tnm142.tsp (EUC_2D, n=142)...
built wspd1.5
Checking Tnm145.tsp (EUC_2D, n=145)...
built wspd1.5
Checking Tnm148.tsp (EUC_2D, n=148)...
built wspd1.5
Checking Tnm151.tsp (EUC_2D, n=151)...
built wspd1.5
Checking Tnm154.tsp (EUC_2D, n=154)...
built w